In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Suma una duración específica a una cadena de fecha y hora y devuelve el resultado en un formato detallado. Esta herramienta convierte la cadena de entrada en un objeto *datetime* de Python, suma la duración especificada en la unidad solicitada y devuelve una cadena formateada con la fecha y hora resultantes. Admite diversas unidades de tiempo —como segundos, minutos, horas, días, semanas, meses y años— y aplica un tratamiento especial a los cálculos de meses y años para tener en cuenta la duración variable de los meses y los años bisiestos. El resultado se presenta siempre en un formato detallado que incluye el día de la semana, el nombre del mes, el día, el año y la hora con indicador AM/PM (por ejemplo: 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "La cadena de fecha y hora de entrada a la que se sumará la duración. Debe tener el formato especificado por el parámetro input_format.",
            },
            "duration": {
                "type": "number",
                "description": "La cantidad de tiempo a agregar al datetime. Puede ser positiva (para fechas futuras) o negativa (para fechas pasadas). Por defecto es 0.",
            },
            "unit": {
                "type": "string",
                "description": "La unidad de tiempo para la duración. Debe ser una de: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', o 'years'. Por defecto es 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "La cadena de formato para analizar el datetime_str, usando los códigos de formato de strptime de Python. Por ejemplo, '%Y-%m-%d' para fechas en formato ISO como '2025-04-03'. Por defecto es '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Crea un recordatorio cronometrado que notificará al usuario a la hora especificada con el contenido proporcionado. Esta herramienta programa una notificación que se entregará al usuario en la marca de tiempo exacta proporcionada. Debe usarse cuando un usuario desea que se le recuerde algo específico en un momento futuro. El sistema de recordatorio almacenará el contenido y la marca de tiempo, luego activará una notificación a través de los canales de notificación preferidos del usuario (alertas móviles, correo electrónico, etc.) cuando llegue la hora especificada. Los recordatorios persisten incluso si se cierra la aplicación o se reinicia el dispositivo. Los usuarios pueden confiar en esta función para recibir notificaciones importantes urgentes, como reuniones, tareas, horarios de medicación o cualquier otra actividad con plazos determinados.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "El texto del mensaje que se mostrará en la notificación del recordatorio. Esto debe contener la información específica sobre la que el usuario quiere ser recordado, como 'Tomar medicamento', 'Unirse a llamada de video con el equipo', o 'Pagar facturas de servicios'.",
            },
            "timestamp": {
                "type": "string",
                "description": "La fecha y hora exacta cuando se debe activar el recordatorio, formateada como un timestamp ISO 8601 (YYYY-MM-DDTHH:MM:SS) o un timestamp de Unix. El sistema maneja todo el procesamiento de husos horarios internamente, asegurando que los recordatorios se activen en el momento correcto sin importar dónde se encuentre el usuario. Los usuarios pueden especificar simplemente el tiempo deseado sin preocuparse por las configuraciones de huso horario.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "El nombre de la herramienta a invocar.",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "Los argumentos para la herramienta, codificados como una cadena JSON",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [ ]:
# get_current_datetime tool function
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format no puede ser vacío")

    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Devuelve la fecha y hora actuales con el formato especificado.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "Una cadena que especifica el formato de la fecha y hora devuelta. Utiliza los códigos de formato `strftime` de Python.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [6]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "Cuál es la hora exacta fomateada a HH:MM:SS?"   
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

messages.append(
    {
        "role": "assistant",
        "content": response.content
    }
)

messages

[{'role': 'user', 'content': 'Cuál es la hora exacta fomateada a HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01VCqm4cDVxFauV9ZWzsy9uq', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [17]:
result = get_current_datetime(**response.content[0].input)

In [15]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[0].id,
                "content": result,
                "is_error": False
            }
        ]
    }
)

messages

[{'role': 'user', 'content': 'Cuál es la hora exacta fomateada a HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01VCqm4cDVxFauV9ZWzsy9uq', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01VCqm4cDVxFauV9ZWzsy9uq',
    'content': '08:47:50',
    'is_error': False}]}]

In [16]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_011CfxHuf2tWyCp5KfY5saHg', container=None, content=[TextBlock(citations=None, text='La hora exacta formateada a HH:MM:SS es: **08:47:50**', type='text')], diagnostics=None, model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=727, output_tokens=26, output_tokens_details=None, server_tool_use=None, service_tier='standard'))